# EDAT: AST Constraint Checking — Full Hybrid Detection System

This notebook implements, tests, and evaluates a **modular AST-based vulnerability detection layer**  
designed to augment transformer models (GraphCodeBERT, CodeBERT, etc.) with structured security signals.

| CWE | Vulnerability | Detection strategy |
|-----|---------------|--------------------|
| CWE-077 | Command Injection | `os.system` / `subprocess` with dynamic args |
| CWE-089 | SQL Injection | SQL keyword + string concatenation or `.execute(var)` |
| CWE-094 | Code Injection | `eval()`/`exec()` with non-constant argument |
| CWE-022 | Path Traversal | File ops with user-controlled path / `../` literal |
| CWE-079 | XSS | Unsafe rendering of user input |
| CWE-352 | CSRF | Route handler with POST and no CSRF decorator |
| CWE-601 | Open Redirect | `redirect(user_input)` |
| unknown | Fallback | No rule matched |


## 0. Setup

In [ ]:
# Standard library only — no pip installs needed for the detector itself
import ast, sys, json, re
from pathlib import Path
from typing import List, Tuple, Dict, Any
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.labelsize': 11, 'font.family': 'DejaVu Sans',
})

# ── Import detector ──────────────────────────────────────────────────────────
# If running from the notebook directory alongside ast_vulnerability_detector.py:
sys.path.insert(0, str(Path.cwd()))
from ast_vulnerability_detector import (
    VulnerabilityDetector,
    scan_code,
    evaluate_detector,
    extract_edat_features,
    extract_identifiers,
    enrich_dataframe,
    ALL_CWES,
    CWE_SEVERITY,
)

print('All imports OK.')
print(f'Python {sys.version}')

## 1. Test Cases — Ground Truth Suite

In [ ]:
# Each entry: (label, true_cwe, source_code)
TEST_CASES: List[Tuple[str, str, str]] = [

    # ── CWE-077 Command Injection ────────────────────────────────────────────
    ("CWE-077 TP — os.system with variable", "CWE-077",
     """
import os
def run_cmd(user_cmd):
    os.system(user_cmd)          # dynamic arg → should flag
"""),

    ("CWE-077 TP — subprocess shell=True", "CWE-077",
     """
import subprocess
def ping(host):
    subprocess.call(host, shell=True)   # shell + dynamic → flag
"""),

    ("CWE-077 FP guard — os.system literal", "unknown",
     """
import os
def run_safe():
    os.system('ls -la')          # constant arg → must NOT flag
"""),

    # ── CWE-089 SQL Injection ────────────────────────────────────────────────
    ("CWE-089 TP — f-string in execute", "CWE-089",
     """
def get_user(username):
    query = f"SELECT * FROM users WHERE name = '{username}'"
    cursor.execute(query)
"""),

    ("CWE-089 TP — concat in execute", "CWE-089",
     """
def search(term):
    sql = "SELECT id FROM items WHERE name = '" + term + "'"
    db.execute(sql)
"""),

    ("CWE-089 FP guard — parameterised query", "unknown",
     """
def get_user_safe(username):
    cursor.execute('SELECT * FROM users WHERE name = %s', (username,))
"""),

    # ── CWE-094 Code Injection ───────────────────────────────────────────────
    ("CWE-094 TP — eval(user_input)", "CWE-094",
     """
def run_expr(user_input):
    result = eval(user_input)    # dynamic → flag
    return result
"""),

    ("CWE-094 TP — exec(code)", "CWE-094",
     """
def execute_code(code):
    exec(code)                   # dynamic → flag
"""),

    ("CWE-094 FP guard — eval literal", "unknown",
     """
def safe_eval():
    result = eval('1 + 1')       # constant → must NOT flag
    return result
"""),

    # ── CWE-022 Path Traversal ───────────────────────────────────────────────
    ("CWE-022 TP — open with request param", "CWE-022",
     """
def read_file(request):
    filename = request.args.get('file')
    with open(filename) as f:    # user-controlled path → flag
        return f.read()
"""),

    ("CWE-022 TP — literal traversal", "CWE-022",
     """
import os
def get_config():
    path = os.path.join(base, '../../../etc/passwd')
    return open(path).read()
"""),

    ("CWE-022 FP guard — safe open", "unknown",
     """
def read_config():
    with open('/etc/app/config.yaml') as f:
        return f.read()          # constant path → must NOT flag
"""),

    # ── CWE-079 XSS ─────────────────────────────────────────────────────────
    ("CWE-079 TP — render_template_string(user_input)", "CWE-079",
     """
from flask import render_template_string, request
def greet():
    name = request.args.get('name')
    return render_template_string('<h1>Hello ' + name + '</h1>')
"""),

    ("CWE-079 TP — Markup(user_input)", "CWE-079",
     """
from markupsafe import Markup
def render(request):
    html = Markup(request.form['content'])   # bypasses escaping → flag
    return html
"""),

    # ── CWE-352 CSRF ─────────────────────────────────────────────────────────
    ("CWE-352 TP — POST route no CSRF", "CWE-352",
     """
from flask import request
@app.route('/transfer', methods=['POST'])
def transfer():
    amount = request.form['amount']
    do_transfer(amount)
"""),

    ("CWE-352 FP guard — CSRF decorator present", "unknown",
     """
from flask import request
from flask_wtf.csrf import csrf_protect
@app.route('/transfer', methods=['POST'])
@csrf_protect
def transfer():
    amount = request.form['amount']
    do_transfer(amount)
"""),

    # ── CWE-601 Open Redirect ────────────────────────────────────────────────
    ("CWE-601 TP — redirect(user url)", "CWE-601",
     """
from flask import redirect, request
def login():
    next_url = request.args.get('next')
    return redirect(next_url)    # user-controlled URL → flag
"""),

    ("CWE-601 FP guard — redirect to constant", "unknown",
     """
from flask import redirect
def logout():
    return redirect('/home')     # constant → must NOT flag
"""),
]

print(f'{len(TEST_CASES)} test cases loaded.')

## 2. Run Evaluation Suite

In [ ]:
rows = []
for label, true_cwe, code in TEST_CASES:
    ev = evaluate_detector(code, true_cwe)
    sc = scan_code(code)
    rows.append({
        'test_name':   label,
        'true_cwe':    true_cwe,
        'predicted':   ev['predicted'],
        'correct':     ev['correct'],
        'confidence':  sc['confidence_score'],
        'n_violations':len(ev['violations']),
        'violations':  ev['violations'],
    })

eval_df = pd.DataFrame(rows)

total   = len(eval_df)
correct = eval_df['correct'].sum()
print(f'\n=== Evaluation Results ===')
print(f'Total cases : {total}')
print(f'Correct     : {correct}  ({correct/total*100:.1f}%)')
print(f'Wrong       : {total-correct}  ({(total-correct)/total*100:.1f}%)')
print()

# Pretty display
display_cols = ['test_name','true_cwe','predicted','correct','confidence','n_violations']
styled = eval_df[display_cols].style.applymap(
    lambda v: 'background-color:#d4edda' if v is True else
              'background-color:#f8d7da' if v is False else '',
    subset=['correct']
).format({'confidence': '{:.3f}'})

try:
    display(styled)
except:
    print(eval_df[display_cols].to_string(index=False))

## 3. Detailed Violation Breakdown

In [ ]:
print(f"{'TEST':<52} {'TRUE':<12} {'PRED':<12} {'OK':<5} {'VIOLATIONS'}")
print('-'*110)
for _, row in eval_df.iterrows():
    ok_str  = '✓' if row['correct'] else '✗'
    viols   = ', '.join(f"{c}@L{l}" for c, l in row['violations']) or '—'
    name    = row['test_name'][:50]
    print(f"{name:<52} {row['true_cwe']:<12} {row['predicted']:<12} {ok_str:<5} {viols}")

## 4. Chart A — Accuracy per CWE Class

In [ ]:
cwe_acc = eval_df.groupby('true_cwe')['correct'].agg(['sum','count'])
cwe_acc['accuracy'] = cwe_acc['sum'] / cwe_acc['count'] * 100
cwe_acc = cwe_acc.sort_values('accuracy')

COLORS = {'CWE-077':'#5B5EA6','CWE-089':'#9B2335','CWE-094':'#C47A3A',
          'CWE-022':'#3E8E41','CWE-079':'#2E6DA4','CWE-352':'#8B3A8B',
          'CWE-601':'#C03B2B','unknown':'#888'}

fig, ax = plt.subplots(figsize=(9, 4.5))
colors  = [COLORS.get(c, '#aaa') for c in cwe_acc.index]
bars    = ax.barh(cwe_acc.index, cwe_acc['accuracy'], color=colors, height=0.5, edgecolor='white')

for bar, (idx, row) in zip(bars, cwe_acc.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{row['accuracy']:.0f}%  ({int(row['sum'])}/{int(row['count'])})",
            va='center', fontsize=10)

ax.axvline(100, color='#3E8E41', lw=1, ls='--', alpha=0.6)
ax.set_xlim(0, 130)
ax.set_xlabel('Detection accuracy (%)')
ax.set_title('Per-CWE detection accuracy on test suite')
plt.tight_layout()
plt.savefig('chart_a_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Chart B — Confidence Score Distribution

In [ ]:
tp_conf = eval_df[eval_df['correct'] == True]['confidence']
fp_conf = eval_df[eval_df['correct'] == False]['confidence']

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(range(len(tp_conf)), sorted(tp_conf), color='#3E8E41', label='Correct',
           s=80, zorder=3, marker='o')
if len(fp_conf):
    ax.scatter(range(len(fp_conf)), sorted(fp_conf), color='#C03B2B', label='Wrong',
               s=80, zorder=3, marker='X')

ax.axhline(0.5, color='gray', lw=1, ls='--', alpha=0.6, label='0.5 threshold')
ax.set_xlabel('Test case rank (sorted by confidence)')
ax.set_ylabel('Confidence score')
ax.set_title('Confidence scores: correct vs wrong predictions')
ax.set_ylim(-0.05, 1.05)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('chart_b_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. EDAT Feature Extraction — ML-Ready Vectors

In [ ]:
print('Extracting EDAT feature vectors for all test cases...\n')

feature_rows = []
for label, true_cwe, code in TEST_CASES:
    feat = extract_edat_features(code)
    feat['test_name'] = label
    feat['true_cwe']  = true_cwe
    feature_rows.append(feat)

feat_df = pd.DataFrame(feature_rows)

# Reorder columns for readability
cwe_cols    = [c for c in feat_df.columns if c.startswith('CWE')]
meta_cols   = ['test_name', 'true_cwe']
stat_cols   = ['num_violations','severity_score','confidence','parse_ok',
               'num_functions','num_variables','num_classes','total_identifiers']

feat_df = feat_df[meta_cols + cwe_cols + stat_cols]

try:
    display(feat_df)
except:
    print(feat_df.to_string(index=False))

## 7. Chart C — Multi-Hot CWE Feature Heatmap

In [ ]:
cwe_cols_present = [c for c in cwe_cols if c in feat_df.columns]
heat_data = feat_df[cwe_cols_present].values.astype(float)
row_labels = [r['true_cwe'] + ' | ' + r['test_name'][:35] for _, r in feat_df.iterrows()]

fig, ax = plt.subplots(figsize=(10, max(5, len(row_labels)*0.45)))
im = ax.imshow(heat_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(cwe_cols_present)))
ax.set_xticklabels(cwe_cols_present, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)

for i in range(len(row_labels)):
    for j in range(len(cwe_cols_present)):
        val = int(heat_data[i, j])
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=8, color='white' if val == 1 else '#555')

ax.set_title('Multi-hot CWE flag matrix (EDAT feature vectors)', pad=12)
plt.tight_layout()
plt.savefig('chart_c_multihot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Chart D — Severity Score vs Confidence

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in feat_df.iterrows():
    color = COLORS.get(row['true_cwe'], '#888')
    ax.scatter(row['severity_score'], row['confidence'],
               color=color, s=90, zorder=3,
               edgecolors='white', linewidths=0.8)

# Legend patches
seen = set()
for _, row in feat_df.iterrows():
    cwe = row['true_cwe']
    if cwe not in seen:
        ax.scatter([], [], color=COLORS.get(cwe,'#888'), label=cwe, s=80)
        seen.add(cwe)

ax.set_xlabel('Severity score (sum of CWE weights)')
ax.set_ylabel('Confidence score')
ax.set_title('Severity vs Confidence per test case (coloured by true CWE)')
ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('chart_d_severity_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Apply to Real Dataset (FINAL_train.jsonl)

In [ ]:
import json

DATA_PATH = r'C:\Users\hende\datasets\FINAL_train.jsonl'

data = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except:
            continue

df = pd.DataFrame(data)
df['code']          = df['raw_lines'].apply(lambda x: '\n'.join(x) if isinstance(x, list) else '')
df['is_vulnerable'] = df['label'].apply(lambda x: int(any(l == 1 for l in x)) if isinstance(x, list) else 0)

print(f'Loaded {len(df):,} snippets')
print(f'Enriching with AST features (this may take a minute)...')

df = enrich_dataframe(df)

print(f'Done. Shape: {df.shape}')
print(f'New columns: {[c for c in df.columns if c.startswith("CWE") or c in ["cwe_prediction","confidence_score","violations","parse_ok"]]}')

## 10. Chart E — Rule Prediction vs Ground-Truth Label

In [ ]:
# Among rows where cwe_id is a known CWE (not 'unknown')
known = df[df['cwe_id'] != 'unknown'].copy()
known['match'] = known['cwe_id'] == known['cwe_prediction']

match_by_cwe = known.groupby('cwe_id')['match'].agg(['sum','count'])
match_by_cwe['precision'] = match_by_cwe['sum'] / match_by_cwe['count'] * 100
match_by_cwe = match_by_cwe.sort_values('precision', ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(match_by_cwe)*0.5)))
bar_colors = ['#C03B2B' if v < 30 else '#C47A3A' if v < 60 else '#3E8E41'
              for v in match_by_cwe['precision']]
bars = ax.barh(match_by_cwe.index, match_by_cwe['precision'],
               color=bar_colors, height=0.5, edgecolor='white')

for bar, (idx, row) in zip(bars, match_by_cwe.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['precision']:.1f}%  (n={int(row['count']):,})",
            va='center', fontsize=9)

ax.axvline(50, color='gray', lw=0.8, ls='--', alpha=0.5)
ax.set_xlim(0, 130)
ax.set_xlabel('Rule match rate (%)')
ax.set_title('AST rule prediction alignment with dataset CWE labels')
plt.tight_layout()
plt.savefig('chart_e_rule_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Export ML-Ready Feature Matrix

In [ ]:
cwe_flag_cols = [c for c in df.columns if c.startswith('CWE-')]

ml_cols = (
    ['cwe_id', 'is_vulnerable', 'cwe_prediction', 'confidence_score', 'parse_ok']
    + cwe_flag_cols
    + ['num_violations', 'severity_score', 'num_functions', 'num_variables',
       'num_classes', 'total_identifiers']
)

ml_cols_present = [c for c in ml_cols if c in df.columns]
ml_df = df[ml_cols_present].copy()

out_path = 'edat_ast_ml_features.csv'
ml_df.to_csv(out_path, index=False)
print(f'Saved {out_path}  →  {ml_df.shape[0]:,} rows × {ml_df.shape[1]} columns')

print('\nColumn types:')
print(ml_df.dtypes)

print('\nSample row (CWE flags + stats):')
try:
    display(ml_df[cwe_flag_cols + ['num_violations','severity_score','confidence_score']].head(3))
except:
    print(ml_df[cwe_flag_cols + ['num_violations','severity_score','confidence_score']].head(3).to_string())

## 12. Detection Logic — Summary

### How each CWE is detected

| CWE | AST nodes used | Key condition | False-positive guard |
|-----|---------------|---------------|----------------------|
| **CWE-077** | `ast.Call` | `os.system` or `subprocess.*` with `shell=True` | Arg must be non-constant (`_is_dynamic`) |
| **CWE-089** | `ast.Call`, `ast.BinOp`, `ast.JoinedStr`, `ast.Assign` | SQL keyword present AND string is built dynamically | Parameterised queries (constant string + separate args) are skipped |
| **CWE-094** | `ast.Call` on `eval`/`exec`/`compile` | First arg is not a `ast.Constant` | Literal `eval('1+1')` is ignored |
| **CWE-022** | `ast.Call` on file/path functions | User input in args OR `../` in a string literal | Constant-only paths are skipped |
| **CWE-079** | `ast.Call` on `render_template_string`, `Markup`, `mark_safe`, `write` | Dynamic content derived from user input | Presence of `escape`/`bleach` in scope lowers risk |
| **CWE-352** | `ast.FunctionDef` decorator list | Route decorator + POST data access + no CSRF decorator | Only triggers when all 3 conditions hold |
| **CWE-601** | `ast.Call` on `redirect`/`HttpResponseRedirect` | First arg contains user-input source | Constant URL redirects (e.g. `redirect('/home')`) are safe |

### Suggestions for improving detection

1. **Taint analysis** — Track user-controlled variables across assignments and function calls (inter-procedural dataflow). Currently `_contains_user_input` only does a shallow AST walk; a full taint tracker would follow assignments.
2. **Call graph construction** — Build a call graph so vulnerabilities in called functions are attributed to callers.
3. **Alias tracking** — `req = request; val = req.form['x']` is currently missed because the alias `req` is not traced.
4. **Type inference** — Know that `cursor` is a DB cursor before checking `.execute()`.
5. **Framework-aware rules** — Django's `ORM.raw()`, SQLAlchemy `text()`, Jinja2 `|safe` filter each need tailored checks.
6. **Sanitiser registry** — Maintain a list of known-safe sanitisers (`bleach.clean`, `html.escape`, `markupsafe.escape`) and suppress flags when they wrap the flow.
